In [1]:
import pandas as pd
import folium

# Load dataset
df = pd.read_csv("../data/raw/crime_dataset_india.csv")

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Normalize city names
df["city"] = df["city"].astype(str).str.strip().str.lower()

print("Total rows:", len(df))
print("Unique cities:", df["city"].unique()[:30])


Total rows: 40160
Unique cities: ['ahmedabad' 'chennai' 'ludhiana' 'pune' 'delhi' 'mumbai' 'surat'
 'visakhapatnam' 'bangalore' 'kolkata' 'ghaziabad' 'hyderabad' 'jaipur'
 'lucknow' 'bhopal' 'patna' 'kanpur' 'varanasi' 'nagpur' 'meerut' 'thane'
 'indore' 'rajkot' 'vasai' 'agra' 'kalyan' 'nashik' 'srinagar' 'faridabad']


In [2]:
df["city"] = (
    df["city"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [3]:
df.columns

Index(['report_number', 'date_reported', 'date_of_occurrence',
       'time_of_occurrence', 'city', 'crime_code', 'crime_description',
       'victim_age', 'victim_gender', 'weapon_used', 'crime_domain',
       'police_deployed', 'case_closed', 'date_case_closed'],
      dtype='object')

In [4]:
from city_coordinates import CITY_COORDINATES
# Map lat/lon using CITY_COORDINATES
# fallback to center of India if city not found
df["latitude"] = df["city"].map(lambda x: CITY_COORDINATES.get(x, (22.5937, 78.9629))[0])
df["longitude"] = df["city"].map(lambda x: CITY_COORDINATES.get(x, (22.5937, 78.9629))[1])

print("Sample coordinates:")
print(df[["city", "latitude", "longitude"]].head())
df = df.dropna(subset=["latitude", "longitude"])
print("Total rows after mapping:", len(df))
df[["city", "latitude", "longitude"]].head()


Sample coordinates:
        city  latitude  longitude
0  ahmedabad   23.0225    72.5714
1    chennai   13.0827    80.2707
2   ludhiana   30.9010    75.8573
3       pune   18.5204    73.8567
4       pune   18.5204    73.8567
Total rows after mapping: 40160


,city,latitude,longitude
0,ahmedabad,23.0225,72.5714
1,chennai,13.0827,80.2707
2,ludhiana,30.9010,75.8573
3,pune,18.5204,73.8567
4,pune,18.5204,73.8567


# CREATE BASE MAP

In [6]:
import os

print(os.getcwd())
print(os.listdir("../outputs"))


C:\Users\Tanish_Gupta\OneDrive\Desktop\ML Projects\ai-crime-hotspot-prediction\notebooks
['crime_counts_with_ai_labels.csv']


In [7]:
import pandas as pd

crime_counts = pd.read_csv("../outputs/crime_counts_with_ai_labels.csv")

crime_counts.head()


,city,crime_count,hotspot_level,latitude,longitude,cluster,ai_hotspot_level
0,ahmedabad,1817,high,23.0225,72.5714,1,low
1,bangalore,3588,high,12.9716,77.5946,2,medium
2,chennai,2493,high,13.0827,80.2707,2,medium
3,delhi,5400,high,28.6139,77.2090,0,high
4,hyderabad,2881,high,17.3850,78.4867,2,medium


In [8]:
import folium
import os

# create map
india_map = folium.Map(location=[22.5937, 78.9629], zoom_start=5)

# add markers for all cities
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=row["city"].title()
    ).add_to(india_map)

# create outputs folder safely
output_dir = os.path.join(os.getcwd(), "outputs")
os.makedirs(output_dir, exist_ok=True)

# save map HTML
html_file = os.path.join(output_dir, "crime_hotspots_india.html")
india_map.save(html_file)

print(f"Map saved successfully: {html_file}")


Map saved successfully: C:\Users\Tanish_Gupta\OneDrive\Desktop\ML Projects\ai-crime-hotspot-prediction\notebooks\outputs\crime_hotspots_india.html


In [9]:
def get_color(level):
    if level == "high":
        return "red"
    elif level == "medium":
        return "orange"
    else:
        return "green"


In [10]:
import folium

india_map = folium.Map(location=[22.5937, 78.9629], zoom_start=5)

for _, row in crime_counts.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8,
        color=get_color(row["ai_hotspot_level"]),
        fill=True,
        fill_color=get_color(row["ai_hotspot_level"]),
        fill_opacity=0.8,
        popup=f"""
        <b>City:</b> {row['city'].title()}<br>
        <b>Crime Count:</b> {row['crime_count']}<br>
        <b>AI Hotspot Level:</b> {row['ai_hotspot_level'].upper()}
        """
    ).add_to(india_map)

india_map


In [11]:
import os

os.makedirs("../outputs", exist_ok=True)
india_map.save("../outputs/ai_crime_hotspots_india.html")

print("AI hotspot map saved successfully")


AI hotspot map saved successfully
